# Getting Started with cande-wrapper

This notebook demonstrates how to use the `cande-wrapper` Python package to run CANDE
(Culvert ANalysis and DEsign) finite element analyses.

**Prerequisites:** The package must be installed with the compiled Fortran extension.
See `README.md` for installation instructions.

```bash
pip install -e ".[dev]"
```

## 1. Import the package

In [1]:
from cande_wrapper import CandeEngine
from pathlib import Path
import shutil
import tempfile

## 2. Set up a working directory

CANDE reads input from `.cid` files and writes output files to the same directory.
We'll copy the included example input file into a temporary working directory.

In [2]:
# Create a temporary working directory
work_dir = Path(tempfile.mkdtemp(prefix="cande_demo_"))
print(f"Working directory: {work_dir}")

# Copy the example input file
example_cid = Path("tests/example_data/MGK-IO.cid")
shutil.copy2(example_cid, work_dir / "MGK-IO.cid")

print(f"Input file: {work_dir / 'MGK-IO.cid'}")

Working directory: C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t
Input file: C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.cid


## 3. Create the engine and verify input

The `CandeEngine` takes a working directory where `.cid` files live.

In [3]:
engine = CandeEngine(work_dir=work_dir)

# Verify the input file exists
print(f"Input file exists: {engine.check_input('MGK-IO')}")

Input file exists: True


## 4. Inspect the input file

CANDE `.cid` files are fixed-format text. Let's look at the first few lines.

In [4]:
cid_text = (work_dir / "MGK-IO.cid").read_text()
for i, line in enumerate(cid_text.splitlines()[:15], 1):
    print(f"{i:3d}: {line}")

  1:                       A-1!!ANALYS   3 0  2 Bandwidth minimizer example, house on stilts                                   
  2:                    A-2.L3!!BASIC         2
  3:                 B-1.Basic!!    1    2      1000         0        10        10         0
  4:                 B-2.Basic!!    0
  5:                    A-2.L3!!BASIC         2
  6:                 B-1.Basic!!    1    2      1000         0        10        10         0
  7:                 B-2.Basic!!    0
  8:                    C-1.L3!!PREP    Non optimum node numbers                                        
  9:                    C-2.L3!!    1    4    0    3    1   11    8    5    1    0    3
 10:                    C-3.L3!!    1    0         0         0                         
 11:                    C-3.L3!!    2    0        10         0                         
 12:                    C-3.L3!!    6    0         0       2.5                         
 13:                    C-3.L3!!    7    0        10     

## 5. Run the analysis

Call `engine.run()` with the file prefix (without `.cid` extension).
This runs the CANDE Fortran engine and returns a `CandeResult` object.

> **Note:** This cell requires the compiled Fortran extension.
> If you haven't built it yet, see `README.md`.

In [5]:
result = engine.run("MGK-IO")
print(result)

CandeResult(prefix='MGK-IO', output=C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.out)


## 6. Inspect the results

The `CandeResult` object provides paths to all output files.

In [6]:
print(f"Output file: {result.output_file}")
print(f"Log file:    {result.log_file}")
print(f"TOC file:    {result.toc_file}")
print()

# List all files created in the working directory
print("All files in working directory:")
for f in sorted(work_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

Output file: C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.out
Log file:    C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.log
TOC file:    C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t\MGK-IO.ctc

All files in working directory:
  MGK-IO.cid (4,649 bytes)
  MGK-IO.ctc (964 bytes)
  MGK-IO.log (760 bytes)
  MGK-IO.out (24,386 bytes)
  MGK-IO_BeamResults.xml (9,471 bytes)
  MGK-IO_MeshGeom.xml (7,123 bytes)
  MGK-IO_MeshResults.xml (7,154 bytes)
  MGK-IO_PLOT1.dat (1,968 bytes)
  MGK-IO_PLOT2.dat (1,751 bytes)


## 7. Read the output report

The `.out` file contains the full CANDE analysis report.

In [7]:
output = result.output_text
lines = output.splitlines()
print(f"Total output lines: {len(lines)}")
print()
print("=== First 50 lines ===")
for line in lines[:50]:
    print(line)

Total output lines: 812

=== First 50 lines ===
>>    Input file: MGK-IO.cid
>>    Output file: MGK-IO.out
>> 
>> 
>>          *** WELCOME TO CANDE-2025 (Version April 2025) ***


           *** WELCOME TO CANDE-2025 (Version April 2025) ***



 MASTER CONTROL AND PIPE-TYPE DATA FOR PROBLEM #  1, (P#1)



          USER TITLE:  Bandwidth minimizer example, house on stilts               


          EXECUTION MODE ..................   ANALYS  

          SOLUTION LEVEL ..................  #3 USER

          METHODOLOGY (LRFD OR SERVICE) ...  SERVICE

          NUMBER OF PIPE-ELEMENT GROUPS ....       2

          MAXIMUM ITERATIONS PER STEP ......      30


>> 
>> 
>>          *** PROBLEM NUMBER  1
>> 
>> 
>>Problem title:  Bandwidth minimizer example, house on stilts
>> 
>> 
>>          EXECUTION MODE ..................   ANALYS
>> 
>>          SOLUTION LEVEL ..................  #3 USER
>> 
>>          METHODOLOGY (LRFD OR SERVICE) ...  SERVICE
>> 
>>          NUMBER OF PIPE-ELEMENT GR

In [8]:
# Check for normal completion
if "NORMAL EXIT FROM CANDE" in output:
    print("Analysis completed successfully.")
else:
    print("WARNING: Normal exit message not found in output.")

Analysis completed successfully.


## 8. Clean up

In [9]:
shutil.rmtree(work_dir)
print(f"Cleaned up {work_dir}")

Cleaned up C:\Users\dane\AppData\Local\Temp\cande_demo_2sx6va1t
